In [2]:
# Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

In [3]:
# Load Dataset

# df = pd.read_csv("dataset/hotel_reviews.csv")
df = pd.read_csv(r"E:\FakeReviewDetection\dataset\clean_reviews.csv")

df.head()

,deceptive,hotel,polarity,source,text,length
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...,572
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...,286
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...,1104
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...,707
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...,384


In [4]:
# Convert Labels

df["deceptive"] = df["deceptive"].map({
    "truthful":1,
    "deceptive":0
})

In [5]:
# Convert Labels

x= df["text"]

y = df["deceptive"]

In [6]:
# Clean Reviews

from utils.preprocess import clean_reviews

x = x.apply(clean_reviews)

In [7]:
# Train-Test Split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
# TF-IDF

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

x_train= tfidf.fit_transform(x_train)

x_test= tfidf.transform(x_test)

In [9]:
# Compare Multiple Models

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Linear SVM":
        LinearSVC(),

    "Multinomial Naive Bayes":
        MultinomialNB(),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
}


In [10]:
# Train and Evaluate

results = []

for name, model in models.items():

    print("="*60)
    print(name)
    print("="*60)

    model.fit(x_train, y_train)

    predictions = model.predict(x_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(y_test, predictions)

    recall = recall_score(y_test, predictions)

    f1 = f1_score(y_test, predictions)

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nClassification Report\n")

    print(classification_report(y_test, predictions))

    print("\nConfusion Matrix\n")

    print(confusion_matrix(y_test, predictions))

    results.append({

        "Model":name,
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 Score":f1,
        "Model Object":model
    })

Logistic Regression
Accuracy : 0.9
Precision: 0.9050632911392406
Recall   : 0.89375
F1 Score : 0.89937106918239

Classification Report

              precision    recall  f1-score   support

           0       0.90      0.91      0.90       160
           1       0.91      0.89      0.90       160

    accuracy                           0.90       320
   macro avg       0.90      0.90      0.90       320
weighted avg       0.90      0.90      0.90       320


Confusion Matrix

[[145  15]
 [ 17 143]]
Linear SVM
Accuracy : 0.8875
Precision: 0.8924050632911392
Recall   : 0.88125
F1 Score : 0.8867924528301887

Classification Report

              precision    recall  f1-score   support

           0       0.88      0.89      0.89       160
           1       0.89      0.88      0.89       160

    accuracy                           0.89       320
   macro avg       0.89      0.89      0.89       320
weighted avg       0.89      0.89      0.89       320


Confusion Matrix

[[143  17]
 [ 19 

In [11]:
# Compare Results

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Model Object
0,Logistic Regression,0.900000,0.905063,0.89375,0.899371,LogisticRegression(max_iter=1000)
1,Linear SVM,0.887500,0.892405,0.88125,0.886792,LinearSVC()
2,Multinomial Naive Bayes,0.887500,0.918919,0.85000,0.883117,MultinomialNB()
3,Random Forest,0.878125,0.880503,0.87500,0.877743,"(DecisionTreeClassifier(max_features='sqrt', r..."


In [12]:
# Save the Best Model

best_model = results_df.iloc[0]["Model Object"]

joblib.dump(best_model,"models/model.pkl")

joblib.dump(tfidf,"models/tfidf.pkl")

print("Best model saved successfully.")

Best model saved successfully.


In [13]:
# Verify Saved Model

loaded_model = joblib.load("models/model.pkl")

loaded_vectorizer = joblib.load("models/tfidf.pkl")

print(type(loaded_model))

<class 'sklearn.linear_model._logistic.LogisticRegression'>


🚀 This is what examiners like to see

Instead of saying:

"I used Logistic Regression."

You can say:

"I evaluated four machine learning algorithms (Logistic Regression, Linear SVM, Multinomial Naive Bayes, and Random Forest). I compared them using Accuracy, Precision, Recall, and F1-score. I selected the best-performing model based on the highest F1-score and deployed it in the Flask application."

That demonstrates a stronger machine learning workflow than training a single model.